# 06. Written rule vs. applied judgement vs. published metrics

PFF Event Data Specification v2.5 (p. 6) documents how pressure is coded:
- **Attempted (A)**: an opponent within **4.57 m** of the carrier, **closing the space with intensity**
- **Pressured (P)**: the opponent closes to within **2.74 m** while still closing, affecting the carrier or their options
- **No pressure**: opponent **static or moving away**
- Radii are guides: they expand with intensity (e.g. 3.66 m at full sprint can be P) and **shrink from behind**

(The specification states the radii in yards — 3 / 5 / 4 yd; they are converted exactly to metres: 2.74 / 4.57 / 3.66 m.)

Because the coding rule is written down, we can implement it on tracking data and ask:
1. How closely do the analysts' labels follow their own written rule?
2. Which part of the rule do they actually apply — distance, closing movement, direction?
3. How sensitive are the velocity-based metrics (Pressing Intensity, arrival time) to velocity input?

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, cohen_kappa_score, confusion_matrix

from football.common import KEY
from football.pressure import PFF_R_PRESSURED, PFF_R_ATTEMPT, PFF_R_SPRINT, defender_table, pass_frames, intercept_prob, pressing_intensity_total, time_to_intercept, fs_arrival_time

OUT = os.path.join("..", "outputs")
FIG = os.path.join(OUT, "figs")
ev = pd.read_csv(os.path.join(OUT, "events.csv.gz"), low_memory=False)
pa = ev.query("event_type == 'PA' and setpiece == 'O' and pressure.notna()", engine="python").copy()
f0 = pass_frames(OUT, pa[KEY])
frames = dict(tuple(f0.groupby(KEY)))
print(f"passes: {len(pa):,}")

passes: 61,028


## 1. Per-defender table for every pass
`closing` = rate at which the defender–carrier distance shrinks (m/s), from smoothed velocities.
`behind` = defender on the far side of the carrier from the attacked goal (cos < 0).

In [2]:
rows = []
for r in pa.itertuples():
    f = frames.get((r.game_id, r.pe_id))
    if f is None:
        continue
    c, D = f[f.player_id == r.actor_id], f[~f.atk]
    if c.empty or len(D) < 5:
        continue
    t = defender_table(c[["x", "y"]].to_numpy(float)[0], c[["vx", "vy"]].to_numpy(float)[0],
                       D[["x", "y"]].to_numpy(float), D[["vx", "vy"]].to_numpy(float),
                       f[f.atk & (f.player_id != r.actor_id) & (f.position != "GK")][["x", "y"]].to_numpy(float))
    rows.append(pd.DataFrame(dict(game_id=r.game_id, pe_id=r.pe_id, player_id=D.player_id.to_numpy(),
                                  label=r.pressure, presser=r.pressure_player_id,
                                  pff_speed=D.speed.to_numpy(), **t)))
DF = pd.concat(rows, ignore_index=True)
DF["behind"] = DF.cos_goal < 0
lab = DF.groupby(KEY).label.first()
print(f"defender rows: {len(DF):,}   passes: {DF.groupby(KEY).ngroups:,}")
# sanity: our smoothed speed vs the speed PFF stores in the event file
ok = DF.pff_speed.notna()
print(f"corr(smoothed speed, PFF speed) = {np.corrcoef(DF.speed[ok], DF.pff_speed[ok])[0, 1]:.3f}")

defender rows: 671,067   passes: 61,012
corr(smoothed speed, PFF speed) = 0.577


## 2. The written rule, implemented
Each defender gets P / A / N; the pass takes the strongest level over defenders.
Variants isolate each ingredient of the rule. Unspecified quantities are parameters:
`c_min` (what counts as "closing"), `behind_scale` (how much the radius shrinks from behind),
sprint extension (3.66 m counts as P when closing faster than 4 m/s).

In [3]:
def rule(DF, r_p=PFF_R_PRESSURED, r_a=PFF_R_ATTEMPT, c_min=None, behind_scale=1.0, sprint=False):
    scale = np.where(DF.behind, behind_scale, 1.0)
    d = DF.dist / scale
    closing = np.ones(len(DF), bool) if c_min is None else DF.closing.to_numpy() > c_min
    P = (d < r_p) & closing
    if sprint:
        P |= (DF.dist.to_numpy() < PFF_R_SPRINT) & (DF.closing.to_numpy() > 4.0)   # sprint example: raw distance
    A = (d < r_a) & closing
    lvl = np.where(P, 2, np.where(A, 1, 0))
    return pd.Series(lvl, index=DF.index).groupby([DF.game_id, DF.pe_id]).max()


VARIANTS = {
    "distance only (2.74 / 4.57 m)": dict(),
    "+ closing > 0": dict(c_min=0.0),
    "+ closing > 1 m/s": dict(c_min=1.0),
    "+ smaller from behind (x0.6)": dict(behind_scale=0.6),
    "full written rule": dict(c_min=0.0, behind_scale=0.6, sprint=True),
}
ORD = {"N": 0, "A": 1, "P": 2}
y_ord = lab.map(ORD)              # L excluded from ordinal agreement
res = []
for name, kw in VARIANTS.items():
    lvl = rule(DF, **kw).reindex(lab.index)
    m = y_ord.notna()
    res.append(dict(rule=name,
                    exact_agreement=(lvl[m] == y_ord[m]).mean(),
                    weighted_kappa=cohen_kappa_score(y_ord[m].astype(int), lvl[m], weights="linear"),
                    kappa_P_vs_rest=cohen_kappa_score(y_ord[m] == 2, lvl[m] == 2),
                    kappa_any_vs_N=cohen_kappa_score(lab.isin(["P", "A", "L"]), lvl > 0)))
rules = pd.DataFrame(res).round(3)
rules

,rule,exact_agreement,weighted_kappa,kappa_P_vs_rest,kappa_any_vs_N
0,distance only (2.74 / 4.57 m),0.693,0.593,0.606,0.584
1,+ closing > 0,0.653,0.460,0.452,0.468
2,+ closing > 1 m/s,0.615,0.314,0.288,0.328
3,+ smaller from behind (x0.6),0.683,0.571,0.564,0.579
4,full written rule,0.646,0.434,0.415,0.452


In [4]:
# confusion matrix of the full written rule vs the analysts (rows = analysts)
lvl = rule(DF, **VARIANTS["full written rule"]).reindex(lab.index).map({0: "N", 1: "A", 2: "P"})
pd.crosstab(lab, lvl, normalize="index").round(3).rename_axis(index="analyst", columns="written rule")

written rule,A,N,P
analyst,,,
A,0.424,0.454,0.122
L,0.367,0.336,0.297
N,0.129,0.834,0.037
P,0.195,0.380,0.425


## 3. What do the analysts actually use?
Share of passes labelled **P** by distance of the nearest defender, split by whether that defender is
closing, static or moving away, and whether they come from in front or behind. If analysts applied the
written rule, closing defenders would be labelled P far more often than static / retreating ones at the same distance.

In [5]:
near = DF.loc[DF.groupby(KEY).dist.idxmin()].copy()
near["dist_bin"] = pd.cut(near.dist, [0, 1, 2, 2.74, 3.66, 4.57, 6, 8, 100])
near["movement"] = pd.cut(near.closing, [-100, -0.5, 0.5, 2, 100],
                          labels=["moving away", "static", "closing", "closing fast"])
tab_move = near.pivot_table(index="dist_bin", columns="movement", values="label",
                            aggfunc=lambda s: (s == "P").mean(), observed=True).round(3)
tab_dir = near.pivot_table(index="dist_bin", columns="behind", values="label",
                           aggfunc=lambda s: (s == "P").mean(), observed=True).round(3)
tab_dir.columns = ["in front", "behind"]
display(tab_move)
tab_dir

movement,moving away,static,closing,closing fast
dist_bin,,,,
"(0.0, 1.0]",0.897,0.899,0.914,0.917
"(1.0, 2.0]",0.801,0.808,0.824,0.855
"(2.0, 2.74]",0.606,0.606,0.628,0.713
"(2.74, 3.66]",0.359,0.371,0.366,0.474
"(3.66, 4.57]",0.242,0.247,0.185,0.246
"(4.57, 6.0]",0.208,0.176,0.110,0.114
"(6.0, 8.0]",0.155,0.149,0.074,0.078
"(8.0, 100.0]",0.107,0.077,0.046,0.063


,in front,behind
dist_bin,,
"(0.0, 1.0]",0.912,0.881
"(1.0, 2.0]",0.821,0.815
"(2.0, 2.74]",0.627,0.668
"(2.74, 3.66]",0.374,0.456
"(3.66, 4.57]",0.204,0.303
"(4.57, 6.0]",0.122,0.266
"(6.0, 8.0]",0.087,0.282
"(8.0, 100.0]",0.058,0.292


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
x = [iv.mid for iv in tab_move.index]
for col in tab_move.columns:
    axes[0].plot(x, tab_move[col], marker="o", label=str(col))
axes[0].axvline(PFF_R_PRESSURED, ls=":", c="grey"); axes[0].axvline(PFF_R_ATTEMPT, ls=":", c="grey")
axes[0].set_title("By movement of nearest defender"); axes[0].set_xlabel("distance to carrier (m)")
axes[0].set_ylabel("share labelled 'pressured' (P)"); axes[0].legend(frameon=False, fontsize=8)
for col in tab_dir.columns:
    axes[1].plot(x, tab_dir[col], marker="o", label=col)
axes[1].axvline(PFF_R_PRESSURED, ls=":", c="grey"); axes[1].axvline(PFF_R_ATTEMPT, ls=":", c="grey")
axes[1].set_title("By direction"); axes[1].set_xlabel("distance to carrier (m)"); axes[1].legend(frameon=False, fontsize=8)
fig.suptitle("Written rule: closing required, 2.74 / 4.57 m radii (dotted), smaller from behind", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_pressure_rule.png"), dpi=200)

### 3b. Robustness: defender speed as stored by PFF (no direction)
Our velocities come from broadcast tracking (speed correlates only r≈0.69 with the `speed` PFF stores in the
event file). Here the nearest defender's *PFF* speed is used instead: if analysts apply the "intensity" clause,
faster defenders should be labelled P more often at the same distance.

In [7]:
near["pff_speed_bin"] = pd.cut(near.pff_speed, [-1, 1, 2, 3, 4, 20],
                               labels=["<1 (static)", "1-2", "2-3", "3-4", ">4 (sprint)"])
tab_speed = near[near.dist < 6].pivot_table(index="dist_bin", columns="pff_speed_bin", values="label",
                                            aggfunc=lambda s: (s == "P").mean(), observed=True).round(3)
tab_speed

pff_speed_bin,<1 (static),1-2,2-3,3-4,>4 (sprint)
dist_bin,,,,,
"(0.0, 1.0]",0.878,0.874,0.874,0.894,0.933
"(1.0, 2.0]",0.699,0.702,0.778,0.803,0.884
"(2.0, 2.74]",0.401,0.538,0.605,0.659,0.730
"(2.74, 3.66]",0.198,0.333,0.371,0.426,0.497
"(3.66, 4.57]",0.201,0.193,0.220,0.248,0.276
"(4.57, 6.0]",0.106,0.117,0.154,0.124,0.155


## 4. Sensitivity of velocity-based metrics
Pressing Intensity and FS arrival time with and without velocities, and with different parameters.
If removing velocity *improves* agreement, either the velocities are noisy or the analysts do not weight movement.

In [8]:
near_keys = lab.index
m = lab.isin(["P", "N"])
y = (lab[m] == "P").astype(int)


def pass_score(fn, agg="max"):
    s = DF.assign(s=fn(DF)).groupby(KEY).s.agg(agg).reindex(near_keys)
    return roc_auc_score(y, s[m])


sens = {"nearest distance": pass_score(lambda d: -d.dist),
        "Pressing Intensity (Eq. 5 total, with velocity)": pass_score(lambda d: -np.log1p(-d.p_int.clip(upper=1 - 1e-12)), "sum"),
        "Andrienko": pass_score(lambda d: d.andr, "sum"),
        "FS arrival (with velocity)": pass_score(lambda d: -d.arrival)}
sens = pd.Series(sens, name="ROC-AUC P vs N").round(3)
sens

nearest distance                                   0.888
Pressing Intensity (Eq. 5 total, with velocity)    0.795
Andrienko                                          0.864
FS arrival (with velocity)                         0.874
Name: ROC-AUC P vs N, dtype: float64

In [9]:
# recompute velocity-based metrics per pass with variants (velocity zeroed / parameters changed)
VAR = {"PI, no velocity": dict(kind="pi", zero=True), "PI, vmax 7.8": dict(kind="pi", vmax=7.8),
       "PI, reaction 0.3 s": dict(kind="pi", reaction=0.3), "FS arrival, no velocity": dict(kind="fs", zero=True)}
scores = {k: [] for k in VAR}
keys = []
for r in pa.itertuples():
    f = frames.get((r.game_id, r.pe_id))
    if f is None:
        continue
    c, D = f[f.player_id == r.actor_id], f[~f.atk]
    if c.empty or len(D) < 5:
        continue
    pc, vc = c[["x", "y"]].to_numpy(float)[0], c[["vx", "vy"]].to_numpy(float)[0]
    P_, V_ = D[["x", "y"]].to_numpy(float), D[["vx", "vy"]].to_numpy(float)
    keys.append((r.game_id, r.pe_id))
    for k, o in VAR.items():
        V = np.zeros_like(V_) if o.get("zero") else V_
        v0 = np.zeros(2) if o.get("zero") else vc
        if o["kind"] == "pi":
            scores[k].append(pressing_intensity_total(intercept_prob(time_to_intercept(pc, v0, P_, V, o.get("reaction", 0.7), o.get("vmax", 12.0)))))
        else:
            scores[k].append(-fs_arrival_time(pc, P_, V).min())
S = pd.DataFrame(scores, index=pd.MultiIndex.from_tuples(keys, names=KEY)).reindex(near_keys)
sens2 = pd.Series({k: roc_auc_score(y, S.loc[m, k]) for k in VAR}, name="ROC-AUC P vs N").round(3)
pd.concat([sens, sens2])

nearest distance                                   0.888
Pressing Intensity (Eq. 5 total, with velocity)    0.795
Andrienko                                          0.864
FS arrival (with velocity)                         0.874
PI, no velocity                                    0.818
PI, vmax 7.8                                       0.815
PI, reaction 0.3 s                                 0.785
FS arrival, no velocity                            0.888
Name: ROC-AUC P vs N, dtype: float64

## Save

In [10]:
rules.to_csv(os.path.join(OUT, "pressure_rule_agreement.csv"), index=False)
tab_move.to_csv(os.path.join(OUT, "pressure_rate_by_distance_movement.csv"))
tab_speed.to_csv(os.path.join(OUT, "pressure_rate_by_distance_pff_speed.csv"))
pd.concat([sens, sens2]).to_csv(os.path.join(OUT, "pressure_velocity_sensitivity.csv"))
print("saved")

saved
